In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
import time

class XYModel:
    def __init__(self, L, J=1.0):
        """
        Initialize the 2D XY model
        
        Parameters:
        L (int): Linear size of the lattice (L x L)
        J (float): Coupling constant (J > 0 for ferromagnetic)
        """
        self.L = L
        self.J = J
        # Initialize spins randomly between 0 and 2π
        self.spins = np.random.uniform(0, 2*np.pi, (L, L))
        
    def energy(self):
        """Calculate the total energy of the system"""
        # Vectorized energy calculation
        sx = np.cos(self.spins)
        sy = np.sin(self.spins)
        
        # Calculate interactions with neighbors (with periodic boundary)
        energy = 0.0
        
        # Horizontal neighbors
        sx_right = np.roll(sx, -1, axis=1)
        sy_right = np.roll(sy, -1, axis=1)
        energy -= self.J * np.sum(sx * sx_right + sy * sy_right)
        
        # Vertical neighbors
        sx_down = np.roll(sx, -1, axis=0)
        sy_down = np.roll(sy, -1, axis=0)
        energy -= self.J * np.sum(sx * sx_down + sy * sy_down)
        
        return energy
    
    def site_energy(self, i, j):
        """Calculate the energy contribution of a single site"""
        # Get the spin angles
        spin = self.spins[i, j]
        sx, sy = np.cos(spin), np.sin(spin)
        
        # Get neighbor indices with periodic boundary conditions
        i_up = (i - 1) % self.L
        i_down = (i + 1) % self.L
        j_left = (j - 1) % self.L
        j_right = (j + 1) % self.L
        
        # Get neighbor spin components
        neighbors = [
            (np.cos(self.spins[i_up, j]), np.sin(self.spins[i_up, j])),
            (np.cos(self.spins[i_down, j]), np.sin(self.spins[i_down, j])),
            (np.cos(self.spins[i, j_left]), np.sin(self.spins[i, j_left])),
            (np.cos(self.spins[i, j_right]), np.sin(self.spins[i, j_right]))
        ]
        
        # Calculate energy contribution
        energy = 0.0
        for nx, ny in neighbors:
            energy -= self.J * (sx * nx + sy * ny)
            
        return energy
    
    def metropolis_step(self, beta):
        """Perform one Monte Carlo step per site using Metropolis algorithm"""
        for _ in range(self.L * self.L):
            # Choose a random site
            i, j = np.random.randint(0, self.L, 2)
            
            # Calculate current energy contribution
            old_energy = self.site_energy(i, j)
            
            # Propose a new spin angle
            new_angle = np.random.uniform(0, 2*np.pi)
            
            # Store the old angle
            old_angle = self.spins[i, j]
            
            # Temporarily update the spin
            self.spins[i, j] = new_angle
            
            # Calculate new energy contribution
            new_energy = self.site_energy(i, j)
            
            # Calculate energy difference
            delta_E = new_energy - old_energy
            
            # Metropolis acceptance criterion
            if delta_E <= 0 or np.random.random() < np.exp(-beta * delta_E):
                # Accept the move (already updated)
                pass
            else:
                # Reject the move and restore the old angle
                self.spins[i, j] = old_angle
    
    def run_simulation(self, beta, num_sweeps, thermalize_sweeps):
        """Run the simulation for a given temperature"""
        # Thermalization phase
        for _ in range(thermalize_sweeps):
            self.metropolis_step(beta)
        
        # Data collection phase
        energies = []
        magnetizations = []
        vortices = []
        stiffnesses = []
        
        for _ in range(num_sweeps):
            self.metropolis_step(beta)
            
            # Collect energy
            energies.append(self.energy() / (self.L * self.L))
            
            # Calculate magnetization
            mx = np.mean(np.cos(self.spins))
            my = np.mean(np.sin(self.spins))
            m = np.sqrt(mx**2 + my**2)
            magnetizations.append(m)

            vortex = self.compute_vorticity()
            v = np.mean(np.abs(vortex))
            vortices.append(v)

            stiffnesses.append(self.spin_stiffness(beta))
        
        return np.mean(energies), np.std(energies), np.mean(np.array(energies)**2), np.mean(magnetizations), np.std(magnetizations), np.mean(np.array(magnetizations)**2), np.mean(vortices), np.mean(stiffnesses), np.std(stiffnesses)
    
    def compute_vorticity(self):
        """Return a 2D array of vorticity per plaquette"""
        vortices = np.zeros((self.L, self.L))
        for i in range(self.L):
            for j in range(self.L):
                # Spins at four corners of a plaquette (counter-clockwise)
                theta = [
                    self.spins[i, j],
                    self.spins[i, (j + 1) % self.L],
                    self.spins[(i + 1) % self.L, (j + 1) % self.L],
                    self.spins[(i + 1) % self.L, j]
                ]
                # Calculate angle differences (modulo 2π)
                dtheta = [(theta[n + 1] - theta[n] + np.pi) % (2 * np.pi) - np.pi for n in range(3)]
                dtheta.append((theta[0] - theta[3] + np.pi) % (2 * np.pi) - np.pi)

                # Sum angle differences; divide by 2π to get winding number
                winding = np.sum(dtheta) / (2 * np.pi)
                vortices[i, j] = np.round(winding)  # Should be 0, ±1 in practice
        return vortices
    
    def spin_stiffness(self, beta):
        """Compute the spin stiffness (helicity modulus) for the current configuration."""
        L = self.L
        J = self.J
        
        # Sum over x-direction bonds (horizontal)
        sum_cos = 0.0
        sum_sin = 0.0
        
        for i in range(L):
            for j in range(L):
                # Right neighbor (x-direction)
                j_right = (j + 1) % L
                delta_theta = self.spins[i, j] - self.spins[i, j_right]
                sum_cos += np.cos(delta_theta)
                sum_sin += np.sin(delta_theta)
        
        # Average per bond
        N_bonds = L * L
        avg_cos = sum_cos / N_bonds
        avg_sin_squared = (sum_sin ** 2) / N_bonds
        
        # Spin stiffness formula
        rho_s = (J * avg_cos) - (J * beta * avg_sin_squared)
        
        return rho_s
    
def find_cross(stiffness, line, temperatures):
    difference = stiffness - line

    # Find index where the sign of the difference changes (i.e., crossing point)
    crossing_indices = np.where(np.diff(np.sign(difference)))[0]

    # Optional: Get the exact crossing point using linear interpolation
    crossings = []
    for idx in crossing_indices:
        x0, x1 = temperatures[idx], temperatures[idx+1]
        y0, y1 = difference[idx], difference[idx+1]
        # Linear interpolation to find more precise x where crossing occurs
        x_cross = x0 - y0 * (x1 - x0) / (y1 - y0)
        y_cross = 2/np.pi * x_cross  # value of the line at crossing (or use either curve)
        crossings.append((x_cross, y_cross))
    return x_cross

        
def visualize_plots(temperatures, energies, energy_errors, magnetizations, mag_errors, specific_heat, susceptibility, vortices, stiffnesses):
    # Plotting results
    fig, axes = plt.subplots(3, 2, figsize=(12, 10))
    
    # Energy vs Temperature
    axes[0, 0].errorbar(temperatures, energies, yerr=energy_errors, fmt='o-')
    axes[0, 0].set_xlabel('Temperature (T)')
    axes[0, 0].set_ylabel('Energy per site')
    axes[0, 0].set_title('Energy vs Temperature')
    axes[0, 0].grid(True)
    
    # Magnetization vs Temperature
    axes[0, 1].errorbar(temperatures, magnetizations, yerr=mag_errors, fmt='o-')
    axes[0, 1].set_xlabel('Temperature (T)')
    axes[0, 1].set_ylabel('Magnetization')
    axes[0, 1].set_title('Magnetization vs Temperature')
    axes[0, 1].grid(True)
    
    # Specific heat (numerical derivative of energy)
    
    axes[1, 0].plot(temperatures, specific_heat, 'o-')
    axes[1, 0].set_xlabel('Temperature (T)')
    axes[1, 0].set_ylabel('Specific Heat (dE/dT)')
    axes[1, 0].set_title('Specific Heat vs Temperature')
    axes[1, 0].grid(True)
    
    # Susceptibility (derivative of magnetization)
    axes[1, 1].plot(temperatures, susceptibility, 'o-')  # Drop last point which may be noisy
    axes[1, 1].set_xlabel('Temperature (T)')
    axes[1, 1].set_ylabel('Susceptibility (-dM/dT)')
    axes[1, 1].set_title('Susceptibility vs Temperature')
    axes[1, 1].grid(True)

    axes[2, 0].plot(temperatures, vortices, 'o-')  # Drop last point which may be noisy
    axes[2, 0].set_xlabel('Temperature (T)')
    axes[2, 0].set_ylabel('density Vortices')
    axes[2, 0].set_title('vortices vs Temperature')
    axes[2, 0].grid(True)


    axes[2, 1].plot(temperatures, stiffnesses, 'o-')  # Drop last point which may be noisy
    line = 2/np.pi*temperatures
    axes[2, 1].plot(temperatures, line, 'r:', label=f'{2/np.pi}·T ')
    axes[2, 1].set_xlabel('Temperature (T)')
    axes[2, 1].set_ylabel('Stiffness')
    axes[2, 1].set_title('Stiffness vs Temperature')
    axes[2, 1].grid(True)

    T_BKT = find_cross(stiffnesses, line, temperatures)

    print(f"Estimated transition temperature from crossing point of Stiffness and the line with coeff 2/pi: T ≈ {T_BKT:.2f}")
    
    plt.tight_layout()
    plt.show()
    plt.close()
    
def visualize_spins(sample_states):
        # Create visualization of sample states with arrows
    fig, axes = plt.subplots(1, len(sample_states), figsize=(5*len(sample_states), 5))
    for i, (temp, model_state) in enumerate(sample_states):
        
        # Create mesh grid for the arrows
        x, y = np.meshgrid(np.arange(0, L), np.arange(0, L))
        
        # Get the spin components
        u = np.cos(model_state.spins)
        v = np.sin(model_state.spins)
        
        
        # Plot the arrows
        axes[i].quiver(
            x, y, u, v,
            pivot='mid',                # Arrows centered on the grid points
            scale=25,                   # Adjust based on how long you want arrows
            scale_units='width',        # Arrow length is relative to figure width
            width=0.005,                # Thickness of the arrow shaft
            headwidth=4,                # Width of arrowhead (won't matter if headlength is tiny)
            headlength=5,               # Length of arrowhead
            headaxislength=4,           # How far the arrowhead extends back to the shaft
            color='black'               # Or use a colormap if visualizing vorticity
        )
        
        axes[i].set_title(f'T = {temp:.2f}')
        axes[i].set_xlim(-1, L)
        axes[i].set_ylim(-1, L)
        axes[i].set_aspect('equal')
        axes[i].set_xticks([])
        axes[i].set_yticks([])
    
    plt.tight_layout()
    plt.show()
    plt.close()


# Function to simulate the XY model at different temperatures
def simulate(L, temperatures, num_points, sweeps, thermalize_sweeps):
    """Simulate the XY model at different temperatures to observe phase transition"""
    
    # Arrays to store results
    energies = np.zeros(num_points)
    energy_errors = np.zeros(num_points)
    energies_sq = np.zeros(num_points)
    magnetizations = np.zeros(num_points)
    mag_errors = np.zeros(num_points)
    mags_sq = np.zeros(num_points)
    vortices = np.zeros(num_points)
    stiffnesses = np.zeros(num_points)
    
    # Create model
    model = XYModel(L)
    
    print(f"Simulating XY model on {L}x{L} lattice")
    
    # Sample states to visualize
    sample_states = []
    sample_temps = temperatures
    sample_indices = [np.abs(temperatures - t).argmin() for t in sample_temps]
    
    # Run simulations
    start_time = time.time()
    for i, T in enumerate(temperatures):
        beta = 1.0 / T
        
        # Print progress
        
        elapsed = time.time() - start_time
        print(f"Progress: {i}/{num_points} temperatures, elapsed time: {elapsed:.1f}s")
        
        # Run simulation
        energy, energy_std, energy_sq, mag, mag_std, mag_sq, vortice, stiffness, stiffness_std = model.run_simulation(beta, sweeps, thermalize_sweeps)
        
        energies[i] = energy
        energy_errors[i] = energy_std / np.sqrt(sweeps)
        energies_sq[i]=energy_sq
        magnetizations[i] = mag
        mag_errors[i] = mag_std / np.sqrt(sweeps)
        mags_sq[i]=mag_sq
        vortices[i]=vortice
        stiffnesses[i] = stiffness
        
        # Save sample state if this is one of our target temperatures
        if i in sample_indices:
            sample_states.append((T, model))
    
    elapsed = time.time() - start_time
    print(f"Simulation completed in {elapsed:.1f} seconds")

    specific_heat = (beta**2) * (energies_sq - energies**2) / (L * L)
    susceptibility = beta * (L * L) * (mags_sq - magnetizations**2)

    visualize_plots(temperatures, energies, energy_errors, magnetizations, mag_errors, specific_heat, susceptibility, vortices, stiffnesses)
    visualize_spins(sample_states)
    
    return temperatures, energies, magnetizations, specific_heat, susceptibility

# Run the main simulation
if __name__ == "__main__":
    # Parameters
    L = 8  # Lattice size
    T_min = 0.1  # Minimum temperature
    T_max = 2.5  # Maximum temperature
    num_points = 15  # Number of temperature points
    sweeps = 2000  # Monte Carlo sweeps per temperature
    thermalize_sweeps = 1000

    Temps = np.linspace(T_min, T_max, num_points)
    
    # Run simulation
    temps, energies, mags, cv, chi = simulate(L, Temps, num_points, sweeps, thermalize_sweeps)
    
    print("Simulation completed!")
    print(f"Theoretical Kosterlitz-Thouless transition temperature is around T_KT ≈ 0.89")
    